# Модель для бота: анкета + история (16 вопросов)
Набор выбран в ноутбуке 12: история даёт +0.0190 AUC (на всех фолдах), профессия 0.0000.
Обучаем на dev. Hold-out используется только как отчёт, выбор набора на нём не делался.

Что здесь:
1. Проверка, что бот считает признаки так же, как обучение (train/serve skew).
2. Обучение, оценка на dev (CV) и hold-out, калибровка, решения при R = 5.
3. Причины отказа для этой модели.
4. Сквозная проверка: ответы пользователя -> вероятность -> решение -> причины, и «что если».
5. Сохранение модели и конфигурации для бота.

Нужны: models/decision_config.json (11), обновлённый src/ (bot_features.py, explain.py).

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

sys.path.insert(0, "..")
from src.bot_features import BOT_FEATURES, answers_to_features
from src.data import load_features, split_dev_holdout
from src.explain import GROUPS, MIN_REASON_SHAP, group_shap, top_reasons
from src.features import ID_COL, TARGET_COL
from src.metrics import calibration_report, calibration_table, ks_stat
from src.models import cv_predict, make_lgbm, to_lgbm

PROCESSED = Path("../data/processed")
MODELS = Path("../models")
REPORTS = Path("../reports")
REPORTS.mkdir(exist_ok=True)
pd.set_option("display.width", 160)

df = load_features(PROCESSED / "train_features.parquet")
dev, hold = split_dev_holdout(df, PROCESSED / "holdout_ids.csv")
y = dev[TARGET_COL]
y_h = hold[TARGET_COL].values
decision = json.loads((MODELS / "decision_config.json").read_text())
R, T = decision["cost_ratio"], decision["threshold"]
print(f"R = {R}, порог = {T}, порог показа причин = {MIN_REASON_SHAP}")

## 1. Совпадает ли расчёт признаков в боте с обучением
Берём реальные строки dev, превращаем их в «ответы пользователя», прогоняем через `answers_to_features`
и сравниваем с признаками, на которых училась модель. Расхождение означало бы ошибку в формуле бота.

In [ ]:
def row_to_answers(r):
    return {
        "income": r.AMT_INCOME_TOTAL, "credit": r.AMT_CREDIT, "annuity": r.AMT_ANNUITY,
        "goods_price": None if pd.isna(r.AMT_GOODS_PRICE) else r.AMT_GOODS_PRICE,
        "income_type": r.NAME_INCOME_TYPE, "education": r.NAME_EDUCATION_TYPE,
        "employment_years": None if pd.isna(r.DAYS_EMPLOYED) else -r.DAYS_EMPLOYED / 365,
        "own_car": r.FLAG_OWN_CAR, "car_age": None if pd.isna(r.OWN_CAR_AGE) else r.OWN_CAR_AGE,
        "own_realty": r.FLAG_OWN_REALTY, "contract_type": r.NAME_CONTRACT_TYPE,
        "bureau_count": r.BUREAU_COUNT, "bureau_active": r.BUREAU_IS_ACTIVE_SUM,
        "bureau_debt": r.BUREAU_AMT_CREDIT_SUM_DEBT_SUM, "bureau_overdue": r.BUREAU_HAS_OVERDUE_SUM > 0,
        "bureau_last_year": r.BUREAU_LAST1Y_COUNT,
    }


sample = dev.sample(2000, random_state=42)
rebuilt = pd.concat([answers_to_features(row_to_answers(r)) for r in sample.itertuples()], ignore_index=True)
orig = sample[BOT_FEATURES].reset_index(drop=True)

problems = {}
for c in BOT_FEATURES:
    if pd.api.types.is_numeric_dtype(orig[c]):
        a, b = orig[c].astype(float).values, rebuilt[c].astype(float).values
        if c == "BUREAU_HAS_OVERDUE_SUM":  # ответ «да/нет» заменяет число кредитов с просрочкой: сравниваем с min(x, 1)
            a = np.minimum(a, 1)
        if not np.allclose(a, b, equal_nan=True, rtol=1e-6):
            problems[c] = float(np.nanmax(np.abs(a - b)))
    elif not (orig[c].astype(str).values == rebuilt[c].astype(str).values).all():
        problems[c] = "категории различаются"
print("Расхождения:", problems if problems else "нет, все 22 признака совпадают")

## 2. Обучение и оценка

In [ ]:
def make_bot_model():
    return make_lgbm(n_estimators=400, learning_rate=0.05, num_leaves=15, min_child_samples=100,
                     colsample_bytree=0.8, subsample=0.8, reg_lambda=3.0)


X_dev = to_lgbm(dev[BOT_FEATURES])
oof, fold_auc = cv_predict(make_bot_model, X_dev, y)
print(f"CV на dev: AUC {fold_auc.mean():.4f} ± {fold_auc.std():.4f}")

model = make_bot_model().fit(X_dev, y)
p_h = model.predict_proba(to_lgbm(hold[BOT_FEATURES]))[:, 1]

In [ ]:
def decide(y_, p, R=R):
    reject = p >= 1 / (1 + R)
    fn = int(((~reject) & (y_ == 1)).sum())
    fp = int((reject & (y_ == 0)).sum())
    loss = (R * fn + fp) / len(y_)
    naive = min(R * y_.mean(), 1 - y_.mean())
    return {"отказов, %": round(reject.mean() * 100, 1),
            "отсеяно дефолтов, %": round(((reject) & (y_ == 1)).sum() / (y_ == 1).sum() * 100, 1),
            "дефолт среди одобренных, %": round(fn / max((~reject).sum(), 1) * 100, 2),
            "экономия, %": round((1 - loss / naive) * 100, 1)}


rows = []
for name, yy, p in [("dev (OOF)", y.values, oof), ("hold-out", y_h, p_h)]:
    auc = roc_auc_score(yy, p)
    rows.append({"выборка": name, "ROC AUC": round(auc, 4), "Gini": round(2 * auc - 1, 4),
                 "PR AUC": round(average_precision_score(yy, p), 4), "KS": round(ks_stat(yy, p), 4),
                 **{k: v for k, v in calibration_report(name, yy, p).items() if k in ("mean pred", "ECE")},
                 **decide(yy, p)})
pd.DataFrame(rows).set_index("выборка")

In [ ]:
# Bootstrap-интервал AUC на hold-out
rng = np.random.default_rng(42)
n = len(y_h)
boot = [roc_auc_score(y_h[idx], p_h[idx]) for idx in (rng.integers(0, n, n) for _ in range(500))]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"AUC на hold-out {roc_auc_score(y_h, p_h):.4f}, 95% ДИ [{lo:.4f}; {hi:.4f}]")
calibration_table(y_h, p_h, bins=10).round(4)

## 3. Причины отказа для модели бота
Групп причин здесь меньше, чем у финальной модели (12 против 28): бот знает о клиенте меньше.

In [ ]:
X_h = to_lgbm(hold[BOT_FEATURES])
sv = model.booster_.predict(X_h, pred_contrib=True)[:, :-1]
reasons = top_reasons(group_shap(sv, X_h.columns), k=3, min_shap=MIN_REASON_SHAP)

refused = np.where(p_h >= T)[0]
print(f"Отказано: {len(refused)} ({len(refused) / len(p_h):.1%})")
print("Сколько причин показано:")
print(pd.Series([len(reasons[i]) for i in refused]).value_counts().sort_index().rename("клиентов"))

counts = pd.Series([t for i in refused for t, _ in reasons[i]]).value_counts()
print("\nЧастота причин среди отказанных, %:")
print((counts / len(refused) * 100).round(1))

for k in range(3):
    vals = np.array([reasons[i][k][1] for i in refused if len(reasons[i]) > k])
    print(f"Причина №{k + 1}: квантили вклада 10% / 50% / 90%: {np.quantile(vals, [0.1, 0.5, 0.9]).round(2)}")

## 4. Сквозная проверка: ответы -> вероятность -> решение -> причины
Именно этот путь пройдёт бот. Смотрим, ведёт ли модель себя разумно на понятных сценариях.

In [ ]:
def assess(answers):
    X = to_lgbm(answers_to_features(answers))
    p = float(model.predict_proba(X)[:, 1][0])
    contrib = model.booster_.predict(X, pred_contrib=True)[:, :-1]
    r = top_reasons(group_shap(contrib, X.columns), k=3, min_shap=MIN_REASON_SHAP)[0]
    return p, p >= T, r


base = {"income": 150_000, "credit": 400_000, "annuity": 20_000, "goods_price": 400_000,
        "income_type": "Working", "education": "Secondary / secondary special", "employment_years": 5,
        "own_car": "N", "car_age": None, "own_realty": "Y", "contract_type": "Cash loans",
        "bureau_count": 2, "bureau_active": 1, "bureau_debt": 60_000, "bureau_overdue": False, "bureau_last_year": 0}

profiles = {
    "типичный клиент": base,
    "высокая нагрузка, просрочки, много новых кредитов": {**base, "income": 90_000, "credit": 900_000, "annuity": 45_000,
        "goods_price": 800_000, "bureau_count": 7, "bureau_active": 4, "bureau_debt": 500_000,
        "bureau_overdue": True, "bureau_last_year": 3},
    "стабильный, без кредитной истории": {**base, "income": 300_000, "credit": 200_000, "annuity": 8_000,
        "goods_price": 200_000, "education": "Higher education", "employment_years": 12,
        "own_car": "Y", "car_age": 3, "bureau_count": 0},
    "пенсионер": {**base, "income_type": "Pensioner", "employment_years": None, "income": 120_000,
        "credit": 250_000, "annuity": 12_000, "goods_price": 250_000},
}
for name, ans in profiles.items():
    p, refuse, rs = assess(ans)
    print(f"--- {name}: вероятность дефолта {p:.1%} -> {'ОТКАЗ' if refuse else 'одобрить'} ---")
    if refuse:
        for text, val in rs:
            print(f"  {text}  (вклад {val:+.2f})")
    print()

In [ ]:
# «Что если»: как меняется вероятность, если менять один ответ
def prob(**kw):
    return assess({**base, **kw})[0]


print("Сумма кредита (платёж и цена товара растут пропорционально):")
for k in [0.5, 1.0, 1.5, 2.0, 3.0]:
    print(f"  x{k}: {prob(credit=400_000 * k, annuity=20_000 * k, goods_price=400_000 * k):.1%}")

print("\nПросрочки в других банках:")
print("  нет:", f"{prob(bureau_overdue=False):.1%}", "| да:", f"{prob(bureau_overdue=True):.1%}")

print("\nСколько кредитов сейчас активно (из 5 всего):")
for k in range(0, 6):
    print(f"  {k}: {prob(bureau_count=5, bureau_active=k):.1%}")

print("\nСтаж, лет:")
for k in [0.5, 1, 3, 5, 10, 20]:
    print(f"  {k}: {prob(employment_years=k):.1%}")

## 5. Сохраняем модель и конфигурацию для бота

In [ ]:
joblib.dump(model, MODELS / "lgbm_bot.joblib")
(MODELS / "features_bot.json").write_text(json.dumps(BOT_FEATURES, ensure_ascii=False, indent=1))
bot_config = {"cost_ratio": R, "threshold": T, "min_reason_shap": MIN_REASON_SHAP, "calibrated": False,
              "model": "models/lgbm_bot.joblib", "features": "models/features_bot.json",
              "cv_auc_dev": round(float(fold_auc.mean()), 4),
              "auc_holdout": round(float(roc_auc_score(y_h, p_h)), 4)}
(MODELS / "decision_config_bot.json").write_text(json.dumps(bot_config, ensure_ascii=False, indent=2))
print("Сохранено: lgbm_bot.joblib, features_bot.json, decision_config_bot.json")

Что смотрим:
1. Нет ли расхождений в пункте 1 (должно быть «нет»; иначе формулы бота нужно править).
2. AUC на hold-out против dev CV (0.7129) и доверительный интервал; калибровка по децилям.
3. Хватает ли причин при отказе (сколько клиентов получили 3, 2, 1 причину) и не доминирует ли одна.
4. Разумны ли сквозные примеры и «что если»: растёт ли риск при росте суммы, просрочках, числе активных кредитов.
   Немонотонность допустима для отдельных признаков (например, срока), но резкие скачки вызывают вопросы.